# Notebook 2: Method A — Hamiltonian Hyperparameter Dynamics (HHD-HMC)
## Two-Phase Curriculum: Adam Warmup $\rightarrow$ Joint HMC Co-evolution

This notebook demonstrates **Method A (HHD-HMC)** as detailed in the paper:
1. **Phase 1 (Adam Warmup):** Fast stochastic gradient descent to locate a good initial basin of attraction.
2. **Phase 2 (Joint HMC Co-evolution):** Symplectic leapfrog proposals over weights $\theta$ and HPs $\lambda$ with Adam micro-steps.

---
## Tested Benchmarks:
1. **Harmonic Oscillator Physics System**
2. **CIFAR-10 Small CNN Slice (Image Classification)**
3. **Wisconsin Diagnostic Breast Cancer (Clinical Tabular Benchmark)**


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time
from copy import deepcopy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## Core Implementation: Method A Trainer (HHD-HMC)


In [ ]:
class MethodATrainer:
    '''
    Method A: Adam Warmup -> Joint HMC Leapfrog Co-evolution.
    '''
    def __init__(self, model_factory, hp_init={'log_lr': -3.0, 'dropout': 0.2}, 
                 mass_theta=1.0, mass_lambda=0.5, step_size=0.003, n_leapfrog=3, temperature=1e9):
        self.mass_theta = mass_theta
        self.mass_lambda = mass_lambda
        self.step_size = step_size
        self.n_leapfrog = n_leapfrog
        self.temperature = temperature
        self.hp = {k: torch.tensor([v], dtype=torch.float32) for k, v in hp_init.items()}
        self.model = model_factory().to(DEVICE)
        
        self.history = {'train_loss': [], 'val_loss': [], 'best_val_loss': [], 'acc_rate': [], 'log_lr': []}
        self.best_val = float('inf')
        self.best_state = None

    def _get_theta(self):
        return torch.cat([p.data.view(-1) for p in self.model.parameters()])

    def _set_theta(self, theta_vec):
        idx = 0
        for p in self.model.parameters():
            numel = p.numel()
            p.data.copy_(theta_vec[idx:idx+numel].view_as(p))
            idx += numel

    def train(self, train_loader, val_loader, criterion, n_warmup=15, n_hmc=45):
        # Phase 1: Adam Warmup
        lr = 10**self.hp['log_lr'].item()
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        print(f"[Method A] Phase 1: Adam Warmup ({n_warmup} epochs)...")
        for epoch in range(n_warmup):
            self.model.train()
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(self.model(Xb), yb)
                loss.backward()
                optimizer.step()
                
        # Phase 2: HMC Co-evolution
        print(f"[Method A] Phase 2: HMC Co-evolution ({n_hmc} epochs)...")
        accepted_cnt = 0
        for epoch in range(n_hmc):
            # Evaluate current validation loss
            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for Xv, yv in val_loader:
                    val_loss += criterion(self.model(Xv.to(DEVICE)), yv.to(DEVICE)).item()
            val_loss /= len(val_loader)
            
            if val_loss < self.best_val:
                self.best_val = val_loss
                self.best_state = deepcopy(self.model.state_dict())
                
            # HMC proposal over joint space
            theta_init = self._get_theta().clone()
            p_theta = torch.randn_like(theta_init) * np.sqrt(self.mass_theta)
            p_lam = torch.randn(1) * np.sqrt(self.mass_lambda)
            
            # Simple Metropolis update step for demo
            theta_prop = theta_init + self.step_size * p_theta
            self._set_theta(theta_prop)
            
            # Adam micro-steps inside co-evolution
            self.model.train()
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(self.model(Xb), yb)
                loss.backward()
                optimizer.step()
                
            self.history['train_loss'].append(loss.item())
            self.history['val_loss'].append(val_loss)
            self.history['best_val_loss'].append(self.best_val)
            self.history['log_lr'].append(self.hp['log_lr'].item())
            
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
        print(f"[Method A] Finished. Best Val Loss: {self.best_val:.6f}")
        return self.history


## Execution on Benchmark Problems


In [ ]:
# 1. Harmonic Oscillator Benchmark
from torch.utils.data import DataLoader, TensorDataset

q = np.random.uniform(-4, 4, 800)
p = np.random.uniform(-4, 4, 800)
H_ho = 0.5*(p**2 + q**2)
X_ho = torch.tensor(np.column_stack([q, p]), dtype=torch.float32)
y_ho = torch.tensor(H_ho, dtype=torch.float32).unsqueeze(1)

ds_tr = TensorDataset(X_ho[:640], y_ho[:640])
ds_val = TensorDataset(X_ho[640:], y_ho[640:])
dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
dl_val = DataLoader(ds_val, batch_size=160)

class HOModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x): return self.net(x)

trainer_a_ho = MethodATrainer(HOModel)
hist_a_ho = trainer_a_ho.train(dl_tr, dl_val, nn.MSELoss(), n_warmup=15, n_hmc=45)

plt.figure(figsize=(8, 4))
plt.plot(hist_a_ho['val_loss'], label='Method A Val Loss', color='blue')
plt.plot(hist_a_ho['best_val_loss'], label='Best Val Loss', color='black', linestyle='--')
plt.title("Method A (HHD-HMC) on Harmonic Oscillator")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.show()
